In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import plotly.express as px
from utilsforecast.evaluation import evaluate
import plotly.io as pio

from utilsforecast.losses import *
from functools import partial
from plotting_utils import (
    plotly_series as plot_series,
)
import torch.nn as nn
import torch

In [3]:
import logging

from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM,
    NHITS,
    RNN,
    MLP,
    BiTCN,
    GRU,
    NBEATS,
    Autoformer,
    TFT,
    TCN,
    DeepAR,
    DLinear,
    TSMixer,
    PatchTST,
)
from neuralforecast.losses.pytorch import MAE

# logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [4]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [5]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [6]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [7]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


In [8]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]

In [9]:
# Minimal custom RNN model for one-step-ahead forecasting (NeuralForecast compatible)
import torch
import torch.nn as nn
from neuralforecast.common._base_model import BaseModel
from neuralforecast.common._modules import MLP
from neuralforecast.losses.pytorch import MAE
from typing import Optional


class CustomModel(BaseModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False  # Direct, not recursive

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        inference_input_size: Optional[int] = None,
        futr_exog_list=None,
        hist_exog_list=None,
        stat_exog_list=None,
        exclude_insample_y=False,
        loss=MAE(),
        valid_loss=None,
        max_steps: int = 1000,
        learning_rate: float = 1e-3,
        num_lr_decays: int = -1,
        early_stop_patience_steps: int = -1,
        val_check_steps: int = 100,
        batch_size=32,
        valid_batch_size: Optional[int] = None,
        windows_batch_size=128,
        inference_windows_batch_size=1024,
        start_padding_enabled=False,
        step_size: int = 1,
        scaler_type: str = "robust",
        random_seed=1,
        drop_last_loader=False,
        alias: Optional[str] = None,
        optimizer=None,
        optimizer_kwargs=None,
        lr_scheduler=None,
        lr_scheduler_kwargs=None,
        dataloader_kwargs=None,
        **trainer_kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            inference_input_size=inference_input_size,
            futr_exog_list=futr_exog_list,
            hist_exog_list=hist_exog_list,
            stat_exog_list=stat_exog_list,
            exclude_insample_y=exclude_insample_y,
            loss=loss,
            valid_loss=valid_loss,
            max_steps=max_steps,
            learning_rate=learning_rate,
            num_lr_decays=num_lr_decays,
            early_stop_patience_steps=early_stop_patience_steps,
            val_check_steps=val_check_steps,
            batch_size=batch_size,
            valid_batch_size=valid_batch_size,
            windows_batch_size=windows_batch_size,
            inference_windows_batch_size=inference_windows_batch_size,
            start_padding_enabled=start_padding_enabled,
            step_size=step_size,
            scaler_type=scaler_type,
            random_seed=random_seed,
            drop_last_loader=drop_last_loader,
            alias=alias,
            optimizer=optimizer,
            optimizer_kwargs=optimizer_kwargs,
            lr_scheduler=lr_scheduler,
            lr_scheduler_kwargs=lr_scheduler_kwargs,
            dataloader_kwargs=dataloader_kwargs,
            **trainer_kwargs,
        )

    def forward(self, windows_batch):
        pass

# Introduction to Attention Mechanisms in Sequence-to-Sequence (Seq2Seq) Models

## Why Do We Need Attention?

Traditional sequence-to-sequence (seq2seq) models, such as LSTM-to-LSTM architectures, encode the entire input sequence into a single fixed-length context vector (the final hidden and cell states of the encoder). The decoder then uses this context to generate the output sequence.

**Limitation:**  
For long input sequences, compressing all information into a single vector can cause the model to "forget" important details, especially those from earlier time steps. This is known as the **information bottleneck** problem.

**Analogy:**  
Imagine reading a long story and then trying to summarize it in one sentence before retelling it. You might miss key details!

---

## What is the Attention Mechanism?

The **attention mechanism** was introduced to help seq2seq models overcome the information bottleneck. Instead of relying solely on the final encoder state, attention allows the decoder to "look back" at all encoder hidden states and decide which parts of the input sequence are most relevant for generating each output step.

**Key Idea:**  
At each decoding step, the model computes a set of weights (called **attention scores**) over all encoder hidden states. These scores determine how much each input time step should contribute to the current output.

---

## How Does Attention Work in LSTM Seq2Seq?

1. **Encoder:**  
    Processes the input sequence and outputs a sequence of hidden states $[h_1, h_2, ..., h_T]$.

2. **Decoder:**  
    At each output time step $t$, the decoder:
    - Computes attention scores for each encoder hidden state.
    - Forms a **context vector** as a weighted sum of encoder hidden states.
    - Uses this context vector (along with its own hidden state) to generate the next output.

### Mathematical Representation

Suppose the encoder produces hidden states $[h_1, h_2, ..., h_T]$ and the decoder is at time step $t$ with hidden state $s_t$.

1. **Compute attention scores:**
    $$
    e_{t,i} = \text{score}(s_t, h_i)
    $$
    where $\text{score}$ is a function (e.g., dot product, MLP) measuring similarity between decoder state $s_t$ and encoder state $h_i$.

2. **Normalize scores (softmax):**
    $$
    \alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{j=1}^T \exp(e_{t,j})}
    $$
    $\alpha_{t,i}$ is the attention weight for encoder time step $i$ at decoder step $t$.

3. **Compute context vector:**
    $$
    c_t = \sum_{i=1}^T \alpha_{t,i} h_i
    $$

4. **Generate output:**
    The decoder uses $c_t$ and $s_t$ to produce the next output.

---

## Why Is Attention Powerful for Time Series Forecasting?

- **Focus on Relevant Inputs:**  
  The model can dynamically select which past time steps are most important for predicting each future value.
- **Improved Long-Term Memory:**  
  No longer limited by a fixed-length context vector; the model can access all encoder states.
- **Interpretability:**  
  Attention weights show which parts of the input sequence the model "attends to" for each prediction.

---

## Real-World Example

Suppose you're forecasting electricity demand for the next day. With attention, the model can focus on:

- The same hour on previous days (to capture daily seasonality)
- Recent unusual spikes (e.g., a holiday)
- Long-term trends

This flexibility leads to more accurate and interpretable forecasts.

---

## Common Beginner Questions

**Q: Does attention replace the LSTM encoder/decoder?**  
*A: No, attention is an additional mechanism that works alongside LSTM (or other RNNs) to improve information flow.*

**Q: Is attention only useful for long sequences?**  
*A: Attention is most beneficial for longer or more complex sequences, but it can improve performance even for moderate-length time series.*

**Q: Can I visualize attention weights?**  
*A: Yes! Attention weights can be plotted to show which input time steps the model focuses on for each prediction.*

---

In the next section, we'll see how to implement an attention-based seq2seq model for electricity load forecasting, and how to visualize attention weights using `plotly`.

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [batch_size, hidden_dim]
        # encoder_outputs: [batch_size, seq_len, hidden_dim]
        batch_size = encoder_outputs.size(0)
        seq_len = encoder_outputs.size(1)

        # Repeat decoder hidden state seq_len times
        decoder_hidden = decoder_hidden.unsqueeze(1).repeat(1, seq_len, 1)
        # Concatenate
        energy = torch.tanh(
            self.attn(torch.cat((decoder_hidden, encoder_outputs), dim=2))
        )
        # Compute attention scores
        attn_scores = self.v(energy).squeeze(2)  # [batch_size, seq_len]
        attn_weights = F.softmax(attn_scores, dim=1)  # [batch_size, seq_len]
        # Weighted sum of encoder outputs
        context = torch.bmm(
            attn_weights.unsqueeze(1), encoder_outputs
        )  # [batch_size, 1, hidden_dim]
        return context, attn_weights

In [13]:
class EncoderLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, x):
        # x: [batch_size, seq_len, input_dim]
        outputs, (hidden, cell) = self.lstm(x)
        # outputs: [batch_size, seq_len, hidden_dim]
        return outputs, hidden, cell

In [16]:
class DecoderLSTMWithAttention(nn.Module):
    def __init__(self, output_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            output_dim + hidden_dim, hidden_dim, num_layers, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.attention = Attention(hidden_dim)

    def forward(self, y_prev, hidden, cell, encoder_outputs):
        # y_prev: [batch_size, 1, output_dim]
        # hidden: [num_layers, batch_size, hidden_dim]
        # encoder_outputs: [batch_size, seq_len, hidden_dim]
        # Use the top layer hidden state for attention
        decoder_hidden = hidden[-1]  # [batch_size, hidden_dim]
        context, attn_weights = self.attention(decoder_hidden, encoder_outputs)
        # Concatenate y_prev and context
        lstm_input = torch.cat(
            (y_prev, context), dim=2
        )  # [batch_size, 1, output_dim + hidden_dim]
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction = self.fc(output)  # [batch_size, 1, output_dim]
        return prediction, hidden, cell, attn_weights

In [17]:
class CustomLSTM2LSTM(CustomModel):
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int = -1,
        input_dim: int = 1,
        hidden_dim: int = 128,
        output_dim: int = 1,
        num_layers: int = 2,
        teacher_forcing_rate: float = 0.5,
        **kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            **kwargs,
        )
        self.example_input_array = (
            {"insample_y": torch.Tensor(self.batch_size, input_size, 1)},
        )
        self.teacher_forcing_rate = teacher_forcing_rate

        self.encoder = EncoderLSTM(input_dim, hidden_dim, num_layers)
        self.decoder = DecoderLSTMWithAttention(output_dim, hidden_dim, num_layers)

    def training_step(self, batch, batch_idx):
        # Set horizon to h_train in case of recurrent model to speed up training
        if self.RECURRENT:
            self.h = self.h_train

        # windows: [Ws, L + h, C, n_series] or [Ws, L + h, C]
        y_idx = batch["y_idx"]

        windows = self._create_windows(batch, step="train")
        original_outsample_y = torch.clone(
            windows["temporal"][:, self.input_size :, y_idx]
        )
        windows = self._normalization(windows=windows, y_idx=y_idx)

        # Parse windows
        (
            insample_y,
            insample_mask,
            outsample_y,
            outsample_mask,
            hist_exog,
            futr_exog,
            stat_exog,
        ) = self._parse_windows(batch, windows)

        windows_batch = dict(
            insample_y=insample_y,  # [Ws, L, n_series]
            insample_mask=insample_mask,  # [Ws, L, n_series]
            outsample_y=outsample_y,  # [Ws, h, n_series]
            futr_exog=futr_exog,  # univariate: [Ws, L, F]; multivariate: [Ws, F, L, n_series]
            hist_exog=hist_exog,  # univariate: [Ws, L, X]; multivariate: [Ws, X, L, n_series]
            stat_exog=stat_exog,
        )  # univariate: [Ws, S]; multivariate: [n_series, S]

        # Model Predictions
        output = self(windows_batch)
        output = self.loss.domain_map(output)

        if self.loss.is_distribution_output:
            y_loc, y_scale = self._get_loc_scale(y_idx)
            outsample_y = original_outsample_y
            distr_args = self.loss.scale_decouple(
                output=output, loc=y_loc, scale=y_scale
            )
            loss = self.loss(y=outsample_y, distr_args=distr_args, mask=outsample_mask)
        else:
            loss = self.loss(
                y=outsample_y, y_hat=output, y_insample=insample_y, mask=outsample_mask
            )

        if torch.isnan(loss):
            print("Model Parameters", self.hparams)
            print("insample_y", torch.isnan(insample_y).sum())
            print("outsample_y", torch.isnan(outsample_y).sum())
            raise Exception("Loss is NaN, training stopped.")

        train_loss_log = loss.detach().item()
        self.log(
            "train_loss",
            train_loss_log,
            batch_size=outsample_y.size(0),
            prog_bar=True,
            on_epoch=True,
        )
        self.train_trajectories.append((self.global_step, train_loss_log))

        self.h = self.horizon_backup

        return loss

    def forward(self, windows_batch):
        encoder_input = windows_batch[
            "insample_y"
        ]  # [batch_size, src_seq_len, input_dim]
        outsample_y = windows_batch.get(
            "outsample_y", None
        )  # [batch_size, h, output_dim]

        batch_size = encoder_input.size(0)
        outputs = []
        attn_weights_list = []

        # Encode the input sequence
        encoder_outputs, hidden, cell = self.encoder(encoder_input)

        # First decoder input: last value of input sequence
        decoder_input = encoder_input[:, -1:, :]  # [batch_size, 1, input_dim]

        for t in range(self.h):
            out, hidden, cell, attn_weights = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs.append(out)
            attn_weights_list.append(attn_weights)
            # Teacher forcing
            if (
                self.training
                and outsample_y is not None
                and torch.rand(1).item() < self.teacher_forcing_rate
            ):
                decoder_input = outsample_y[:, t : t + 1, :]
            else:
                decoder_input = out  # Use own prediction as next input

        outputs = torch.cat(outputs, dim=1)  # [batch_size, tgt_len, output_dim]
        # Optionally, return attn_weights_list for visualization
        return outputs

In [20]:
horizon = 48

# Try different hyperparmeters to improve accuracy.
models = [
    CustomLSTM2LSTM(
        input_size=2 * horizon,
        h=horizon,  # Forecast horizon
        max_steps=300,  # Number of steps to train
        scaler_type="standard",  # Type of scaler to normalize data
        val_check_steps=10,
        input_dim=1,
        hidden_dim=64,  # Defines the size of the hidden state of the LSTM
        output_dim=1,  # Output dimension of the decoder
        num_layers=2,  # Number of layers in the LSTM
        teacher_forcing_rate=0.5,
    ),
]

nf = NeuralForecast(models=models, freq="30m")
y_hat = nf.cross_validation(
    data,
    step_size=48,
    n_windows=1,
    val_size=96,
).drop("cutoff")

Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name         | Type                     | Params | Mode  | In sizes                                             | Out sizes                                       
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
0 | loss         | MAE                      | 0      | train | ?                                                    | ?                                               
1 | padder_train | ConstantPad1d            | 0      | train | ?                                                    | ?                                               
2 | scaler       | TemporalNorm             | 0      | train | ?                                                    | ?                                               
3 | encoder      | EncoderLST

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

In [21]:
fig = plot_series(y_hat, y_hat)
fig.show()

evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

unique_id,metric,CustomLSTM2LSTM
str,str,f64
"""MAC000193""","""mae""",0.43677
"""MAC000193""","""mse""",0.423759
"""MAC000193""","""rmse""",0.650968
"""MAC000193""","""mape""",5.160767
"""MAC000193""","""smape""",0.733333
"""MAC000193""","""mase""",2.527815
